# Phase 3 · Banking Knowledge Extraction

**Goal**: Learn how the real banking system behaves from cleaned data.
No synthetic data is generated in this phase — we extract and persist
structured knowledge that drives all subsequent generation phases.

**Inputs** (`data/interim/`):
- `transactions.parquet` — canonical transactions
- `accounts.parquet` — canonical accounts

**Outputs** (`data/interim/knowledge_base/`):
| File | Contents |
|---|---|
| `behavior_profiles.parquet` | Per-account statistical profiles |
| `institution_mapping.json` | Institution → branch → city hierarchy |
| `branch_mapping.json` | Branch → city + institution |
| `country_mapping.json` | Country frequencies + risk scores |
| `graph_statistics.json` | Graph-level metrics (degree, PageRank, etc.) |
| `amount_distribution.json` | Log-normal fit params |
| `temporal_distribution.json` | Hour/weekday/month weights |
| `currency_distribution.json` | Currency frequency |
| `payment_type_distribution.json` | Payment type frequency |

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 1 · Imports & Environment Setup
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('phase3_knowledge')

# ── Resolve project root ──────────────────────────────────────────────────────
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
    KAGGLE_MODE  = True
else:
    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while not (PROJECT_ROOT / 'AGENTS.md').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
    KAGGLE_MODE = False

sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
KB_DIR      = INTERIM_DIR / 'knowledge_base'

print(f'Project root : {PROJECT_ROOT}')
print(f'Kaggle mode  : {KAGGLE_MODE}')
print(f'Interim dir  : {INTERIM_DIR}')
print('\n✓ Imports complete')

## 1. Load Cleaned Data

In [ ]:
%%time
tx  = pd.read_parquet(INTERIM_DIR / 'transactions.parquet')
acc = pd.read_parquet(INTERIM_DIR / 'accounts.parquet')

print(f'transactions.parquet : {tx.shape[0]:,} rows × {tx.shape[1]} cols')
print(f'accounts.parquet     : {acc.shape[0]:,} rows × {acc.shape[1]} cols')
print(f'\nFraud rate: {tx["is_fraud"].mean():.4%}')
print(f'Columns: {list(tx.columns[:10])} ...')

## 2. Run Knowledge Extractor

In [ ]:
%%time
from src.generation.core.knowledge_extractor import KnowledgeExtractor

extractor = KnowledgeExtractor(PROJECT_ROOT)
extractor.extract(tx, acc)
extractor.save()

print('\n✓ Knowledge extraction complete')
print(f'Files saved to: {KB_DIR}')
for f in sorted(KB_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')

## 3. Customer Behavior Profiles

In [ ]:
profiles = extractor.behavior_profiles
print(f'Profiles for {len(profiles):,} accounts')
profiles.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Customer Behavior Distributions', fontsize=14, fontweight='bold')

if 'amount_mean' in profiles.columns:
    axes[0].hist(profiles['amount_mean'].clip(upper=profiles['amount_mean'].quantile(0.95)),
                 bins=40, color='#4C72B0', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Mean Tx Amount (NPR) per Account')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

if 'tx_count' in profiles.columns:
    axes[1].hist(profiles['tx_count'].clip(upper=profiles['tx_count'].quantile(0.99)),
                 bins=40, color='#DD8452', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Transaction Count per Account')

if 'cross_border_rate' in profiles.columns:
    axes[2].hist(profiles['cross_border_rate'], bins=30, color='#55A868', edgecolor='white', linewidth=0.5)
    axes[2].set_title('Cross-Border Rate per Account')

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

## 4. Institution & Geographic Knowledge

In [ ]:
import json

with open(KB_DIR / 'institution_mapping.json') as f:
    inst_map = json.load(f)

print(f'Institutions learned: {len(inst_map)}')
for inst, data in sorted(inst_map.items(), key=lambda x: -x[1].get('frequency',0))[:10]:
    n_branches = len(data.get('branches', {}))
    print(f'  {inst:<20} freq={data.get("frequency",0):5d}  branches={n_branches}')

In [ ]:
with open(KB_DIR / 'country_mapping.json') as f:
    country_map = json.load(f)

city_freq = country_map.get('city_frequencies', {})
if city_freq:
    cities = sorted(city_freq.items(), key=lambda x: -x[1])[:15]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh([c[0] for c in cities], [c[1] for c in cities], color='#4C72B0')
    ax.set_xlabel('Account Count')
    ax.set_title('Top 15 Cities by Account Count')
    ax.invert_yaxis()
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print('No city frequency data available')

## 5. Transaction Distributions

In [ ]:
with open(KB_DIR / 'amount_distribution.json') as f:
    amt_dist = json.load(f)

print('Amount Distribution Parameters:')
for k, v in amt_dist.items():
    if not isinstance(v, list):
        print(f'  {k:<25} = {v:,.2f}')

# Plot log-normal fit vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount Distribution', fontsize=13, fontweight='bold')

amounts = tx['amount_npr'].dropna().clip(lower=1)
axes[0].hist(amounts.clip(upper=amounts.quantile(0.99)), bins=60,
             color='#4C72B0', edgecolor='white', linewidth=0.5, density=True)
axes[0].set_title('Amount Distribution (NPR)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

log_amounts = np.log(amounts)
axes[1].hist(log_amounts, bins=60, color='#DD8452', edgecolor='white', linewidth=0.5, density=True)
from scipy.stats import norm as scipy_norm
mu, sigma = amt_dist.get('log_mean', 11), amt_dist.get('log_std', 2.5)
x = np.linspace(log_amounts.min(), log_amounts.max(), 200)
axes[1].plot(x, scipy_norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'Log-Normal fit (μ={mu:.2f}, σ={sigma:.2f})')
axes[1].set_title('Log(Amount) with Fitted Distribution')
axes[1].legend()

for ax in axes:
    ax.set_ylabel('Density')
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 6. Graph Knowledge Summary

In [ ]:
with open(KB_DIR / 'graph_statistics.json') as f:
    graph_stats = json.load(f)

print('Transaction Graph Statistics')
print('=' * 40)
for k, v in graph_stats.items():
    if not isinstance(v, list):
        print(f'  {k:<35} = {v}')

print(f'\nTop 5 Fan-Out accounts (high out-degree):')
for acct, degree in graph_stats.get('top_fanout_accounts', [])[:5]:
    print(f'  Account {acct}: out-degree = {degree}')

print(f'\nTop 5 Fan-In accounts (high in-degree):')
for acct, degree in graph_stats.get('top_fanin_accounts', [])[:5]:
    print(f'  Account {acct}: in-degree = {degree}')

## 7. Temporal Patterns

In [ ]:
with open(KB_DIR / 'temporal_distribution.json') as f:
    temporal = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Temporal Transaction Patterns', fontsize=13, fontweight='bold')

hour_w = temporal.get('hour_weights', {})
if hour_w:
    hours = [int(h) for h in sorted(hour_w.keys(), key=int)]
    weights = [hour_w[str(h)] for h in hours]
    axes[0].bar(hours, weights, color='#4C72B0', edgecolor='white')
    axes[0].set_title('Transaction Volume by Hour')
    axes[0].set_xlabel('Hour of Day')
    axes[0].set_ylabel('Proportion')

dow_w = temporal.get('dow_weights', {})
if dow_w:
    dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dows = [int(d) for d in sorted(dow_w.keys(), key=int)]
    weights = [dow_w[str(d)] for d in dows]
    colors = ['#DD8452' if d >= 5 else '#4C72B0' for d in dows]
    axes[1].bar([dow_labels[d] for d in dows], weights, color=colors, edgecolor='white')
    axes[1].set_title('Transaction Volume by Day of Week')
    axes[1].set_ylabel('Proportion')

month_w = temporal.get('month_weights', {})
if month_w:
    months = [int(m) for m in sorted(month_w.keys(), key=int)]
    weights = [month_w[str(m)] for m in months]
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    axes[2].bar([month_labels[m-1] for m in months], weights, color='#55A868', edgecolor='white')
    axes[2].set_title('Transaction Volume by Month')
    axes[2].set_ylabel('Proportion')
    axes[2].tick_params(axis='x', rotation=45)

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## Summary

✅ **Phase 3 complete.** The knowledge base captures:
- Customer behavioral profiles for every real account
- Institution/branch/city hierarchy for constraint-aware generation
- Country risk mapping from Nepal Rastra Bank context
- Log-normal amount distribution parameters (with quantiles)
- Temporal weights for realistic timestamp generation
- Graph topology metrics for fraud pattern injection

**Next step →** Phase 4: Synthetic Account Generation (`phase4_generate_5m.ipynb`)